<a href="https://colab.research.google.com/github/cc-huang-0716/internship_tests/blob/main/timeseries_handcraft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# data
from google.colab import drive
import pandas as pd
from sklearn.model_selection import train_test_split

"""
主要擬想用於偵測餘額異常性，
但編碼者沒有觀看銀行內部數據的權限，
所以再麻煩各位進行應用上微調，
本處僅以合成數據示例，並編碼出手刻模型架構
數據清理以及實際應用方面再麻煩主要應用者們，謝謝。
若有問題，再請聯絡我，" huang.07160716@gmail.com "
"""

drive.mount('/content/drive')
anomaly_df = pd.read_csv("/content/drive/MyDrive/peak_series_data.csv")
normal_df = pd.read_csv("/content/drive/MyDrive/stable_series_500.csv")
merge_df = pd.concat([anomaly_df, normal_df])
#print(anomaly_df)
#print(normal_df)
#print(merge_df)
train_df, test_df = train_test_split(merge_df, test_size=0.3, shuffle=True, random_state=42)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ADFuller Test
"""
對每個樣本建立一個 AR 模型，
判斷自變數的係數 (Beta) 是否大-於1。
若小於 1, 資料符合穩態。 可以直接往下做時序模型。
如果大於等於 1, 表示有單位根。又稱 " 偽回歸 " 問題。
表示殘差不隨樣本數擴大而影響衰弱，又稱 " 隨機漫步 " 問題。
後續可能要採取二階差分等方式整理原始數據再進行檢定.
如果想要自動獲取最佳的 p_lag 值，需另撰寫以 AIC，BIC為損失函數的最佳化流程，
目前以 p_lag = 1 撰寫。
目前訓練兩天下來，ROC_AUC : 0.63( 2025 / 08 / 06 )
"""
import numpy as np
import pandas as pd

def adfuller_test(your_dataframe, threshold):

  stationary_dict = dict()
  for index, row in your_dataframe.iterrows():
    row = row.values.astype(float)
    dy = np.diff(row)
    y_lag = row[:-1]
    X = np.column_stack([np.ones(len(y_lag)), y_lag])
    y = dy

    try:
        XTX_inv = np.linalg.pinv(X.T @ X)
        beta_hat = XTX_inv @ X.T @ y # OLS 的封閉解
    except np.linalg.LinAlgError:
        print(f"Sample {index}: Singular matrix, cannot compute coefficients.")
        continue

    residuals = y - X @ beta_hat
    RSS = np.sum(residuals**2)
    n = len(y)
    k = X.shape[1]

    if n - k <= 0:
        print(f"Sample {index}: Not enough data points to perform t-test.")
        continue

    sigma2 = RSS / (n - k)
    se_beta = np.sqrt(np.diag(sigma2 * XTX_inv))

    t_beta = beta_hat[1] / se_beta[1]

    print(f"Sample {index}, t_beta = {t_beta:.4f}")

    if abs(t_beta) > threshold: # H0
        print("無單位根，平穩")
        stationary_dict[index] = row
    else:             # Ha
        print("有單位根，不平穩")

  return stationary_dict

#adfuller_test(your_dataframe=merge_df, threshold=2.57) #threshold 可調

Sample 0, t_beta = -2.4532
有單位根，不平穩
Sample 1, t_beta = -2.8755
無單位根，平穩
Sample 2, t_beta = -2.8826
無單位根，平穩
Sample 3, t_beta = -2.9018
無單位根，平穩
Sample 4, t_beta = -4.4465
無單位根，平穩
Sample 5, t_beta = -4.3339
無單位根，平穩
Sample 6, t_beta = -3.0626
無單位根，平穩
Sample 7, t_beta = -4.0991
無單位根，平穩
Sample 8, t_beta = -2.8062
無單位根，平穩
Sample 9, t_beta = -2.9355
無單位根，平穩
Sample 10, t_beta = -4.2211
無單位根，平穩
Sample 11, t_beta = -2.9832
無單位根，平穩
Sample 12, t_beta = -2.8292
無單位根，平穩
Sample 13, t_beta = -3.0573
無單位根，平穩
Sample 14, t_beta = -2.8487
無單位根，平穩
Sample 15, t_beta = -2.3976
有單位根，不平穩
Sample 16, t_beta = -2.2609
有單位根，不平穩
Sample 17, t_beta = -2.3903
有單位根，不平穩
Sample 18, t_beta = -4.2766
無單位根，平穩
Sample 19, t_beta = -2.7996
無單位根，平穩
Sample 20, t_beta = -4.0679
無單位根，平穩
Sample 21, t_beta = -4.2452
無單位根，平穩
Sample 22, t_beta = -2.1783
有單位根，不平穩
Sample 23, t_beta = -3.0712
無單位根，平穩
Sample 24, t_beta = -2.3396
有單位根，不平穩
Sample 25, t_beta = -2.8974
無單位根，平穩
Sample 26, t_beta = -4.2559
無單位根，平穩
Sample 27, t_beta = -2.2307
有單位根

{1: array([0.03010772, 0.11425557, 0.3682305 , 0.60950943, 0.33293047,
        0.15123184, 0.31812213, 0.39130075, 0.52166516, 1.28210183,
        1.38539587, 1.66744784, 1.89951901, 2.07047274, 2.07269283,
        2.22553687, 2.10506947, 2.04219241, 1.96132802, 1.9711957 ,
        2.43897721, 2.09528202, 2.19162303, 1.97369922, 1.82901197,
        1.98609266, 2.05685421, 1.88088839, 1.76903364, 1.93637046,
        1.7896848 , 1.7881138 , 1.80101839, 1.63684024, 2.11438504,
        2.23381598, 1.7875126 , 1.80873417, 1.69702345, 1.83932389,
        1.63970872, 1.6289458 , 1.74219158, 1.88999546, 1.62638426,
        1.57108651, 1.40369323, 1.2026542 , 1.51982284, 1.59014682,
        1.35351541, 1.61085561, 2.07816983, 2.27666595, 1.97184115,
        1.82486787, 2.07732444, 1.92135762, 2.02625743, 2.13982269,
        1.98040392, 2.0451358 , 1.39144432, 1.20665238, 1.19064595,
        0.92102829, 1.25871518, 0.97331652, 0.89019143, 0.87768905,
        1.16716922, 0.9048967 , 1.21008664, 1

In [ ]:
# ADFuller Test
"""
對每個樣本建立一個 AR 模型，
判斷自變數的係數 (Beta) 是否大-於1。
若小於 1, 資料符合穩態。 可以直接往下做時序模型。
如果大於等於 1, 表示有單位根。又稱 " 偽回歸 " 問題。
表示殘差不隨樣本數擴大而影響衰弱，又稱 " 隨機漫步 " 問題。
後續可能要採取二階差分等方式整理原始數據再進行檢定.
如果想要自動獲取最佳的 p_lag 值，需另撰寫以 AIC，BIC為損失函數的最佳化流程，
目前以 p_lag = 1 撰寫。
目前訓練兩天下來，ROC_AUC : 0.63( 2025 / 08 / 06 )
"""
import numpy as np
import pandas as pd

def adfuller_test_with_BIC(your_dataframe):

  for index, row in your_dataframe.iterrows():
    row = row.values.astype(float)
    dy = np.diff(row)
    y_lag = row[:-1]
    X = np.column_stack([np.ones(len(y_lag)), y_lag])
    y = dy

    try:
        XTX_inv = np.linalg.pinv(X.T @ X)
        beta_hat = XTX_inv @ X.T @ y # OLS 的封閉解
    except np.linalg.LinAlgError:
        print(f"Sample {index}: Singular matrix, cannot compute coefficients.")
        continue

    residuals = y - X @ beta_hat
    RSS = np.sum(residuals**2)
    n = len(y)
    k = X.shape[1]

    if n - k <= 0:
        print(f"Sample {index}: Not enough data points to perform t-test.")
        continue

    sigma2 = RSS / (n - k)
    se_beta = np.sqrt(np.diag(sigma2 * XTX_inv))

    t_beta = beta_hat[1] / se_beta[1]
    bic = n * np.log(RSS / n) + k * np.log(n) # 不用常數項

  return float(t_beta), float(bic), int(n)

def minimize_bic():
  best_params = {t_beta = }

In [ ]:
"""
製作 ARCH 的 MLE 公式
"""
import scipy.special
import math
import numpy as np

def mle_process(v, m, T):
  for idx, row in df.iterrows():
    value_list = []
    row_values = row.values
    T = len(row)
    var = np.var(row_values[m+1:T]) # m 表 ARCH 的階數
    for value in row:
      log_likelihood =

In [ ]:
import numpy as np
import pandas as pd

your_DataFrame = sta_pd
order = 4
#def variance(seq):
#    i_list = []
#    for i in seq:
#        mean = sum(seq)/len(seq)
#        j=(i - mean)**2
#        i_list.append(j)
#    var = sum(i_list)/len(seq)
#    return np.asarray(i_list)

# 先估計均值(按照上次測試結果，以能最佳化 MSE 的 AR(4) 估計)
# 不設常數項
stationary_dict = adfuller_test(your_dataframe=merge_df, threshold=2.57)
stationary_list = list(stationary_dict.values())
#print(stationary_list)

sta_pd = pd.DataFrame(stationary_list)
#print(sta_pd)
your_coef = 0.15
phi = np.array([your_coef ** i for i in range(1, 5)]) # 預設值，可更動(記得不要弄到發散)/記得基數(your_coef)要一樣

try:
  if sum(phi) > 1:
    print("你的序列會導致結果發散，請保證總和小於 1 !!!")
except:
  print("係數序列小於1")

y_pred = []
results = []
for _, row in sta_pd.iterrows():
    row_values = row.values
    epsilon = np.random.randn(len(row_values))
    y = np.array(row.values)
    y[:order] = row_values[:order]
    mean = sum(row_values)/len(row_values)
    const = mean * (1 - sum(phi)) # mean_reverse, 保證數值在期望值上下浮動
    for i in range(4, len(row_values)):
      y[i] = const + np.dot(phi, y[i-order:i][::-1]) + epsilon[i]
    results.append(y)

y_true_list = []
for _, row in your_DataFrame.iterrows():
  y_true = row.values
  y_true_list.append(y_true)

y_true_arr = np.array(y_true_list)
y_pred_arr = np.array(results)

if y_true_arr.shape[0] != y_pred_arr.shape[0]:
  print(f"欄位維度不對稱，y_true是{y_true_arr.shape[0]}, y_pred是{y_pred_arr.shape[0]}")
elif y_true_arr.shape[1] != y_pred_arr.shape[1]:
  print(f"列的維度不對稱，y_true是{y_true_arr.shape[1]}, y_pred是{y_pred_arr.shape[1]}")
else:
  x = y_true_arr - y_pred_arr # epsilon_arr

# 用 leverage 衡量殘差，判斷樣本是否為離群值 (異常值)

hat_matrix = x @ np.linalg.pinv(x.T @ x) @ x.T
h_arr = np.diag(hat_matrix)


ab_or_not = dict()
for e in h_arr:
  if abs(e) >= 4 / x.shape[1]:
    ab_or_not[e]=1 #異常
  else:
    ab_or_not[e]=0 #正常
results_pd = pd.DataFrame(list(ab_or_not.items()), columns=["value", "abnormal"])
print(results_pd)

Sample 0, t_beta = -2.4532
有單位根，不平穩
Sample 1, t_beta = -2.8755
無單位根，平穩
Sample 2, t_beta = -2.8826
無單位根，平穩
Sample 3, t_beta = -2.9018
無單位根，平穩
Sample 4, t_beta = -4.4465
無單位根，平穩
Sample 5, t_beta = -4.3339
無單位根，平穩
Sample 6, t_beta = -3.0626
無單位根，平穩
Sample 7, t_beta = -4.0991
無單位根，平穩
Sample 8, t_beta = -2.8062
無單位根，平穩
Sample 9, t_beta = -2.9355
無單位根，平穩
Sample 10, t_beta = -4.2211
無單位根，平穩
Sample 11, t_beta = -2.9832
無單位根，平穩
Sample 12, t_beta = -2.8292
無單位根，平穩
Sample 13, t_beta = -3.0573
無單位根，平穩
Sample 14, t_beta = -2.8487
無單位根，平穩
Sample 15, t_beta = -2.3976
有單位根，不平穩
Sample 16, t_beta = -2.2609
有單位根，不平穩
Sample 17, t_beta = -2.3903
有單位根，不平穩
Sample 18, t_beta = -4.2766
無單位根，平穩
Sample 19, t_beta = -2.7996
無單位根，平穩
Sample 20, t_beta = -4.0679
無單位根，平穩
Sample 21, t_beta = -4.2452
無單位根，平穩
Sample 22, t_beta = -2.1783
有單位根，不平穩
Sample 23, t_beta = -3.0712
無單位根，平穩
Sample 24, t_beta = -2.3396
有單位根，不平穩
Sample 25, t_beta = -2.8974
無單位根，平穩
Sample 26, t_beta = -4.2559
無單位根，平穩
Sample 27, t_beta = -2.2307
有單位根

In [ ]:
# arch
"""
ARCH模型假設:
(1) 假設誤差是i.i.d
(2) 模型是對於誤差的波動偵測，並認為當期誤差和前一期誤差平方有關
(3) 1982年作者論文網站 "http://www.econ.uiuc.edu/~econ536/Papers/engle82.pdf"
!!! 有些論文中很精細的定義跟條件我暫時沒有加入，有需要可以到上面網址參考觀察
"""
import numpy as np
import pandas as pd

#def variance(seq):
#    i_list = []
#    for i in seq:
#        mean = sum(seq)/len(seq)
#        j=(i - mean)**2
#        i_list.append(j)
#    var = sum(i_list)/len(seq)
#    return np.asarray(i_list)

# 先估計均值(按照上次測試結果，以能最佳化 MSE 的 AR(4) 估計)
# 不設常數項
stationary_dict = adfuller_test(your_dataframe=merge_df, threshold=2.57)
stationary_list = list(stationary_dict.values())
#print(stationary_list)

sta_pd = pd.DataFrame(stationary_list)
#print(sta_pd)

def AR_with_residual_test_process(order, your_coef, your_DataFrame):
  phi = np.array([your_coef ** i for i in range(1, 5)]) # 預設值，可更動(記得不要弄到發散)/記得基數(your_coef)要一樣

  try:
    if sum(phi) > 1:
      print("你的序列會導致結果發散，請保證總和小於 1 !!!")
  except:
    print("係數序列小於1")

  y_pred = []
  results = []
  for _, row in sta_pd.iterrows():
      row_values = row.values
      epsilon = np.random.randn(len(row_values))
      y = np.array(row.values)
      y[:order] = row_values[:order]
      mean = sum(row_values)/len(row_values)
      const = mean * (1 - sum(phi)) # mean_reverse, 保證數值在期望值上下浮動
      for i in range(4, len(row_values)):
        y[i] = const + np.dot(phi, y[i-order:i][::-1]) + epsilon[i]
      results.append(y)

  y_true_list = []
  for _, row in your_DataFrame.iterrows():
    y_true = row.values
    y_true_list.append(y_true)

  y_true_arr = np.array(y_true_list)
  y_pred_arr = np.array(results)

  if y_true_arr.shape[0] != y_pred_arr.shape[0]:
    print(f"欄位維度不對稱，y_true是{y_true_arr.shape[0]}, y_pred是{y_pred_arr.shape[0]}")
  elif y_true_arr.shape[1] != y_pred_arr.shape[1]:
    print(f"列的維度不對稱，y_true是{y_true_arr.shape[1]}, y_pred是{y_pred_arr.shape[1]}")
  else:
    x = y_true_arr - y_pred_arr # epsilon_arr

  # 用 leverage 衡量殘差，判斷樣本是否為離群值 (異常值)

  hat_matrix = x @ np.linalg.pinv(x.T @ x) @ x.T
  h_arr = np.diag(hat_matrix)


  ab_or_not = dict()
  for e in h_arr:
    if abs(e) >= 4 / x.shape[1]:
      ab_or_not[e]=1 #異常
    else:
      ab_or_not[e]=0 #正常
  results_pd = pd.DataFrame(list(ab_or_not.items()), columns=["value", "abnormal"])
  return results_pd
print(AR_process(order=4, your_coef=0.15, your_DataFrame=sta_pd)) #shape=(n, 2)
#__________________________________________________________________________________________________________________________________________________
# ARCH

Sample 0, t_beta = -2.4532
有單位根，不平穩
Sample 1, t_beta = -2.8755
無單位根，平穩
Sample 2, t_beta = -2.8826
無單位根，平穩
Sample 3, t_beta = -2.9018
無單位根，平穩
Sample 4, t_beta = -4.4465
無單位根，平穩
Sample 5, t_beta = -4.3339
無單位根，平穩
Sample 6, t_beta = -3.0626
無單位根，平穩
Sample 7, t_beta = -4.0991
無單位根，平穩
Sample 8, t_beta = -2.8062
無單位根，平穩
Sample 9, t_beta = -2.9355
無單位根，平穩
Sample 10, t_beta = -4.2211
無單位根，平穩
Sample 11, t_beta = -2.9832
無單位根，平穩
Sample 12, t_beta = -2.8292
無單位根，平穩
Sample 13, t_beta = -3.0573
無單位根，平穩
Sample 14, t_beta = -2.8487
無單位根，平穩
Sample 15, t_beta = -2.3976
有單位根，不平穩
Sample 16, t_beta = -2.2609
有單位根，不平穩
Sample 17, t_beta = -2.3903
有單位根，不平穩
Sample 18, t_beta = -4.2766
無單位根，平穩
Sample 19, t_beta = -2.7996
無單位根，平穩
Sample 20, t_beta = -4.0679
無單位根，平穩
Sample 21, t_beta = -4.2452
無單位根，平穩
Sample 22, t_beta = -2.1783
有單位根，不平穩
Sample 23, t_beta = -3.0712
無單位根，平穩
Sample 24, t_beta = -2.3396
有單位根，不平穩
Sample 25, t_beta = -2.8974
無單位根，平穩
Sample 26, t_beta = -4.2559
無單位根，平穩
Sample 27, t_beta = -2.2307
有單位根

In [ ]:
import pandas as pd
import numpy as np

results_df = AR_process(order = 4, your_coef = 0.15, your_DataFrame = sta_pd)
#print(results_df)

def drop_non_ar_value(order, df):
  results_list = []
  for _, row in df.iterrows():
    row_values = row.values
    order = int(order)
    new_row = row_values[order:]
    results_list.append(new_row)

  results_array = np.asarray(results_list, dtype=float)
  return results_array

sta_array = drop_non_ar_value(order=4, df=sta_pd)
results_array = drop_non_ar_value(order=4, df=results_df)

epsilon_array = sta_array - results_array
#print(epsilon_array)

# Engle's ARCH Test
"""
檢查是否有條件異質變異數
H0是變異數恆定，Ha則是隨時間改變
"""
engle_arr = np.square(epsilon_array)
def engle_lm_test(q, engle_arr, results_df):
# 滯後項 q 選5
  q = 5
  e_array = []
  for row in engle_arr:
    e_list = []
    for i in range(q, len(row)):
      y = row[i]
      y = np.array(y)
      y = np.reshape(y, (1, 1))

      try:
        x = row[i:i+4]
        #j = 0
        if len(x) != 4:
          #j += 1
          break
      except:
        continue

      x = np.array(x)
      x = np.reshape(x, (1, 4))
      const = np.ones(shape=1) # 常數項設 1
      cons = np.reshape(const, (1, 1))
      X = np.concatenate([cons, x], axis=1)
      XTX_inv = np.linalg.pinv(X.T @ X) # 避免奇異值矩陣問題，所以採用 pinv 而不是 inv
      beta = XTX_inv @ X.T @ y
      epsilon_t = (X @ beta).item()
      e_list.append(epsilon_t)

    e_array.append(e_list)

  e_arr = []
  for row in engle_arr:
    new_row = row[q-1:len(row)-q+1]
    e_arr.append(new_row)
  e_arr = np.array(e_arr, dtype=float) # 對應預測值的真實值陣列
  e_array = np.array(e_array, dtype=float) #預測值陣列

  try:
    arch_list = []
    non_arch_list = []
    rss = np.sum((e_arr - e_array) ** 2, axis=1)
    sst = np.sum(e_arr - np.mean(e_arr), axis=1)
    R2 = 1 - rss / sst
    LM = len(row) * R2   # 檢定量
    LM = np.reshape(LM, (110,1))
    results_df = np.array(results_df)
    LM_results_df = np.concatenate([LM, results_df], axis=1)
    for row in LM_results_df:
      if row[0] >= 11.0705:  # 這是個右尾檢定，閥值服從卡方，自由度為 q ，目前使用 95% 信心水準。
        arch_list.append(row)
        print("該樣本的變異數恆定，可執行ARCH。")
      else:
        non_arch_list.append(row)
        print("該樣本變異數不恆定，不可執行ARCH")
  except:
    print("檢定流程有錯誤")

  if len(non_arch_list) + len(arch_list) != LM_results_df.shape[0]:
    print("尚有樣本未經分類或分類錯誤")
  else:
    print("分類完成")
    print(f"arch_list有{len(arch_list)}個樣本")
    print(f"non_arch_list有{len(non_arch_list)}個樣本")
  return arch_list, non_arch_list

該樣本的變異數恆定，可執行ARCH。
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本的變異數恆定，可執行ARCH。
該樣本的變異數恆定，可執行ARCH。
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本的變異數恆定，可執行ARCH。
該樣本的變異數恆定，可執行ARCH。
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本的變異數恆定，可執行ARCH。
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本的變異數恆定，可執行ARCH。
該樣本變異數不恆定，不可執行ARCH
該樣本的變異數恆定，可執行ARCH。
該樣本的變異數恆定，可執行ARCH。
該樣本的變異數恆定，可執行ARCH。
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本的變異數恆定，可執行ARCH。
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本的變異數恆定，可執行ARCH。
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本的變異數恆定，可執行ARCH。
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本的變異數恆定，可執行ARCH。
該樣本的變異數恆定，可執行ARCH。
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本的變異數恆定，可執行ARCH。
該樣本的變異數恆定，可執行ARCH。
該樣本的變異數恆定，可執行ARCH。
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可執行ARCH
該樣本變異數不恆定，不可

In [ ]:
import numpy as np
import pandas as pd

class arch_or_not:
  def __init__(self, order, sta_array, results_array, epsilon_array):
    self.order = order
    self.sta_array = sta_array
    self.results_array = results_array
    self.epsilon_array = epsilon_array

  def drop_non_ar_value(self):
    results_list = []
    for _, row in df.iterrows():
      row_values = row.values
      order = int(order)
      new_row = row_values[order:]
      results_list.append(new_row)

    results_array = np.asarray(results_list, dtype=float)
    return results_array

  def engle_lm_test(self, q):
    q = 5   # 滯後項 q 選5
    sta_array = drop_non_ar_value(order=4, df=sta_pd)
    results_array = drop_non_ar_value(order=4, df=results_df)
    epsilon_array = sta_array - results_array
    e_array = []
    for row in engle_arr:
      e_list = []
      for i in range(q, len(row)):
        y = row[i]
        y = np.array(y)
        y = np.reshape(y, (1, 1))

        try:
          x = row[i:i+4]
          #j = 0
          if len(x) != 4:
            #j += 1
            break
        except:
          continue

        x = np.array(x)
        x = np.reshape(x, (1, 4))
        const = np.ones(shape=1) # 常數項設 1
        cons = np.reshape(const, (1, 1))
        X = np.concatenate([cons, x], axis=1)
        XTX_inv = np.linalg.pinv(X.T @ X) # 避免奇異值矩陣問題，所以採用 pinv 而不是 inv
        beta = XTX_inv @ X.T @ y
        epsilon_t = (X @ beta).item()
        e_list.append(epsilon_t)

      e_array.append(e_list)

    e_arr = []
    for row in engle_arr:
      new_row = row[q-1:len(row)-q+1]
      e_arr.append(new_row)
    e_arr = np.array(e_arr, dtype=float) # 對應預測值的真實值陣列
    e_array = np.array(e_array, dtype=float) #預測值陣列

    try:
      arch_list = []
      non_arch_list = []
      rss = np.sum((e_arr - e_array) ** 2, axis=1)
      sst = np.sum(e_arr - np.mean(e_arr), axis=1)
      R2 = 1 - rss / sst
      LM = len(row) * R2   # 檢定量
      LM = np.reshape(LM, (110,1))
      results_df = np.array(results_df)
      LM_results_df = np.concatenate([LM, results_df], axis=1)
      for row in LM_results_df:
        if row[0] >= 11.0705:  # 這是個右尾檢定，閥值服從卡方，自由度為 q ，目前使用 95% 信心水準。
          arch_list.append(row)
          print("該樣本的變異數恆定，可執行ARCH。")
        else:
          non_arch_list.append(row)
          print("該樣本變異數不恆定，不可執行ARCH")
    except:
      print("檢定流程有錯誤")

    if len(non_arch_list) + len(arch_list) != LM_results_df.shape[0]:
      print("尚有樣本未經分類或分類錯誤")
    else:
      print("分類完成")
      print(f"arch_list有{len(arch_list)}個樣本")
      print(f"non_arch_list有{len(non_arch_list)}個樣本")
    return arch_list, non_arch_list

output = arch_or_not(order=4, sta_array=sta_array, results_array=results_array, epsilon_array=epsilon_array)

In [ ]:
def ARCH_AR_process(lag ,order, your_coef, your_DataFrame):
  phi = np.array([your_coef ** i for i in range(1, 5)]) # 預設值，可更動(記得不要弄到發散)

  if sum(phi) > 1:
    print("你的序列會導致結果發散，請保證總和小於 1 !!!")
  else:
    print("係數序列小於1")

  results = []

  for _, row in your_DataFrame.iterrows():
      row_values = row.values
      y = np.array(row.values)
      y[:order] = row_values[:order]
      mean = sum(row_values)/len(row_values)
      const = mean * (1 - sum(phi))  # mean_reverse
      for i in range(4, len(row_values)):
        y[i] = const + np.dot(phi, y[i-order:i][::-1]) + epsilon[i]
        results.append(y[i])

  results_pd = pd.DataFrame(results)
  return results_pd

In [ ]:
results = arch_process(lag = 4 ,order = 3, your_coef = 0.15, your_DataFrame = sta_pd)

係數序列小於1


NameError: name 'alpha1' is not defined